# Federated RAG System with LLM Query Routing

This notebook demonstrates how to use the Federated RAG system with intelligent query routing across 8 specialized client domains.

## System Overview

- 8 Federated Clients with domain-specific knowledge bases
- LLM-Based Query Routing for intelligent client selection
- RRF Aggregation for combining results from multiple clients
- GPT-4o-mini for response generation

## 1. Setup and Initialization

Import required libraries and load environment variables.

In [21]:
from dotenv import load_dotenv
from src.core.embeddings.embedding_handler import EmbeddingHandler
from src.core.generator.response_generator import ResponseGenerator
from src.core.utils.helpers import load_config
from src.federated.client import FederatedClient
from src.federated.server import FederatedServer

load_dotenv()

print("Libraries imported successfully")

Libraries imported successfully


## 2. Initialize Federated System

Load configuration and initialize all 8 federated clients.

In [22]:
config = load_config("config/federated_config.yaml")

print("Configuration loaded:")
print(f"  - Number of clients: {config['system']['num_clients']}")
print(f"  - Embedding model: {config['embedding']['model']}")
print(f"  - LLM model: {config['llm']['model']}")
print(f"  - Query routing: {config['routing']['enabled']}")
print(f"  - Aggregation strategy: {config['retrieval']['aggregation_strategy']}")

Configuration loaded:
  - Number of clients: 8
  - Embedding model: sentence-transformers/all-MiniLM-L6-v2
  - LLM model: gpt-4o-mini
  - Query routing: True
  - Aggregation strategy: rrf


In [23]:
print("Initializing embedding handler...")
embedding_handler = EmbeddingHandler(config['embedding']['model'])
print("Embedding handler ready")

Initializing embedding handler...
Model loaded. Embedding dimension: 384
Embedding handler ready


In [24]:
print("\nInitializing federated clients...\n")

clients = []
for i, client_config in enumerate(config['clients'], 1):
    print(f"[{i}/{len(config['clients'])}] Initializing {client_config['name']}...")
    
    client = FederatedClient(
        client_id=client_config['id'],
        documents_path=client_config['data_path'],
        embedding_handler=embedding_handler,
        config=config
    )
    client.index_documents()
    clients.append(client)

print(f"\nAll {len(clients)} clients initialized successfully!")


Initializing federated clients...

[1/8] Initializing Company Documents...
[client_1] Initializing...
Collection 'client_1_collection' initialized with 1016 documents
[client_1] Indexing documents...


Batches: 100%|██████████| 2/2 [00:00<00:00, 10.24it/s]


Added 44 documents to the collection
[client_1] Indexed 44 chunks from 4 documents
[2/8] Initializing Employee Manuals...
[client_2] Initializing...
Collection 'client_2_collection' initialized with 1778 documents
[client_2] Indexing documents...


Batches: 100%|██████████| 3/3 [00:00<00:00, 10.15it/s]


Added 77 documents to the collection
[client_2] Indexed 77 chunks from 4 documents
[3/8] Initializing Product Information...
[client_3] Initializing...
Collection 'client_3_collection' initialized with 1786 documents
[client_3] Indexing documents...


...

Added 60 documents to the collection
[client_6] Indexed 60 chunks from 3 documents
[7/8] Initializing Finance & Accounting...
[client_7] Initializing...
Collection 'client_7_collection' initialized with 664 documents
[client_7] Indexing documents...


Batches: 100%|██████████| 3/3 [00:00<00:00,  7.07it/s]


Added 83 documents to the collection
[client_7] Indexed 83 chunks from 3 documents
[8/8] Initializing Marketing & Communications...
[client_8] Initializing...
Collection 'client_8_collection' initialized with 616 documents
[client_8] Indexing documents...


Batches: 100%|██████████| 3/3 [00:00<00:00,  7.53it/s]

Added 77 documents to the collection
[client_8] Indexed 77 chunks from 2 documents

All 8 clients initialized successfully!


In [25]:
print("Initializing federated server...")
server = FederatedServer(clients, embedding_handler, config)

generator = ResponseGenerator(
    model=config['llm']['model'],
    temperature=config['llm']['temperature'],
    max_tokens=config['llm']['max_tokens']
)

print("\nFederated RAG system ready!")

Initializing federated server...
[Server] Query routing ENABLED (max 3 clients per query)

[Server] Initialized with 8 clients
[Server] Using aggregation strategy: rrf

Federated RAG system ready!


## 3. Client Domain Overview

Let's see what domains each client specializes in.

In [26]:
import pandas as pd

client_overview = [
    {"Client ID": "client_1", "Domain": "Company Documents", "Topics": "Policies, Safety, Code of Conduct"},
    {"Client ID": "client_2", "Domain": "Employee Manuals", "Topics": "Benefits, PTO, HR Policies"},
    {"Client ID": "client_3", "Domain": "Product Information", "Topics": "Features, Technical Specs, APIs"},
    {"Client ID": "client_4", "Domain": "Customer Success & Sales", "Topics": "Onboarding, Support, Training"},
    {"Client ID": "client_5", "Domain": "Engineering & Operations", "Topics": "SDLC, DevOps, Incident Response"},
    {"Client ID": "client_6", "Domain": "Legal & Compliance", "Topics": "GDPR, Contracts, Regulations"},
    {"Client ID": "client_7", "Domain": "Finance & Accounting", "Topics": "Expenses, Budgets, AP Procedures"},
    {"Client ID": "client_8", "Domain": "Marketing & Communications", "Topics": "Brand, Social Media, Content"}
]

df = pd.DataFrame(client_overview)
df

,Client ID,Domain,Topics
0,client_1,Company Documents,"Policies, Safety, Code of Conduct"
1,client_2,Employee Manuals,"Benefits, PTO, HR Policies"
2,client_3,Product Information,"Features, Technical Specs, APIs"
3,client_4,Customer Success & Sales,"Onboarding, Support, Training"
4,client_5,Engineering & Operations,"SDLC, DevOps, Incident Response"
5,client_6,Legal & Compliance,"GDPR, Contracts, Regulations"
6,client_7,Finance & Accounting,"Expenses, Budgets, AP Procedures"
7,client_8,Marketing & Communications,"Brand, Social Media, Content"


## 4. Helper Function for Queries

Create a helper function to query the system and display results nicely.

In [27]:
def ask_question(question, show_sources=True):
    """
    Ask a question to the federated RAG system
    
    Args:
        question: The question to ask
        show_sources: Whether to display source documents
    """
    print("="*80)
    print(f"QUESTION: {question}")
    print("="*80)
    
    docs = server.federated_retrieve(question)
    
    client_ids = set(doc.get('client_id', 'unknown') for doc in docs)
    print(f"\nRetrieved {len(docs)} documents from {len(client_ids)} clients: {', '.join(sorted(client_ids))}")
    
    formatted_docs = [
        {
            'document': d['content'], 
            'metadata': d.get('metadata', {}),
            'distance': d.get('distance', 0.0)
        } 
        for d in docs
    ]
    
    answer = generator.generate_response(question, formatted_docs)
    
    print(f"\nANSWER:")
    print(answer)
    
    if show_sources and docs:
        print(f"\nSOURCES:")
        for i, doc in enumerate(docs[:3], 1):
            source = doc.get('metadata', {}).get('source', 'Unknown')
            client_id = doc.get('client_id', 'unknown')
            print(f"   {i}. {source} (from {client_id})")
    
    print("\n" + "="*80 + "\n")
    return answer, docs

## 6. Example Queries - Cross-Domain

These queries span multiple domains and may be routed to several clients.

### Query 1: Company Documents (Client 1)

In [28]:
answer, docs = ask_question("What is the company's workplace safety policy?")

QUESTION: What is the company's workplace safety policy?

[Server] Query: 'What is the company's workplace safety policy?'
[Server] Routed to: client_1
[Server] Querying 1 clients...
  [client_1] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_1] Retrieved 5 chunks

Retrieved 5 documents from 1 clients: client_1

ANSWER:
The company's workplace safety policy focuses on emergency procedures, specifically for fire emergencies. The key points of the policy include:

1. Activating the nearest fire alarm pull station in case of fire.
2. Evacuating the building immediately using the nearest exit.
3. Not using elevators during evacuation.
4. Proceeding to the designated assembly point.
5. Not re-entering the building until authorized by the fire department.

The designated fire assembly point for the North Building is specified as "North." 

This information is consistent across all provided documents.

SOURCES:
   1. safety_manual.txt (from client

### Query 2: Employee Manuals (Client 2)

In [29]:
answer, docs = ask_question("How many vacation days do employees get per year?")

QUESTION: How many vacation days do employees get per year?

[Server] Query: 'How many vacation days do employees get per year?'
[Server] Routed to: client_2
[Server] Querying 1 clients...
  [client_2] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_2] Retrieved 5 chunks

Retrieved 5 documents from 1 clients: client_2

ANSWER:
The provided context does not specify the number of vacation days employees get per year. It only mentions a maximum carryover of 5 days to the following year and that unused vacation is paid out upon termination.

SOURCES:
   1. employee_handbook.txt (from client_2)
   2. employee_handbook.txt (from client_2)
   3. employee_handbook.txt (from client_2)




### Query 3: Product Information (Client 3)

In [30]:
answer, docs = ask_question("What are the main features of the product?")

QUESTION: What are the main features of the product?

[Server] Query: 'What are the main features of the product?'
[Server] Routed to: client_3
[Server] Querying 1 clients...
  [client_3] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_3] Retrieved 5 chunks

Retrieved 5 documents from 1 clients: client_3

ANSWER:
The main features of the product are:

**Dashboard and Analytics:**
- Real-time infrastructure monitoring
- Customizable dashboards with drag-and-drop widgets
- Advanced analytics and reporting
- Predictive cost optimization
- Automated alerts and notifications
- Historical data retention: 2 years

**Automation Tools:**
- Infrastructure as Code (IaC) support
- Automated backup and disaster recovery
- Policy-based governance
- Scheduled task automation

SOURCES:
   1. product_catalog.txt (from client_3)
   2. product_catalog.txt (from client_3)
   3. product_catalog.txt (from client_3)




### Query 4: Customer Success (Client 4)

In [31]:
answer, docs = ask_question("What is the customer onboarding process?")

QUESTION: What is the customer onboarding process?

[Server] Query: 'What is the customer onboarding process?'
[Server] Routed to: client_4
[Server] Querying 1 clients...
  [client_4] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_4] Retrieved 5 chunks

Retrieved 5 documents from 1 clients: client_4

ANSWER:
The customer onboarding process includes the following steps:

1. **Scheduling the Kick-off Meeting**:
   - Schedule within 24 hours of contract signature.
   - Attendees: Customer Success (CS) Manager, Implementation Specialist, and Customer stakeholders.
   - Agenda for the meeting:
     - Welcome and introductions
     - Review customer goals and success criteria
     - Confirm technical requirements
     - Present the onboarding timeline (30/60/90 day plan)
     - Assign action items with owners
     - Schedule the next check-in.

2. **Technical Setup (Days 1-7)**:
   - Provision accounts and licenses.
   - Configure initial setting

## 6. Example Queries - Cross-Domain

These queries span multiple domains and should be routed to multiple clients.

In [32]:
answer, docs = ask_question("What training is required for new employees?")

QUESTION: What training is required for new employees?

[Server] Query: 'What training is required for new employees?'
[Server] Routed to: client_2, client_1, client_4
[Server] Querying 3 clients...
  [client_1] Found 5 results
  [client_2] Found 5 results
  [client_4] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_1] Retrieved 2 chunks
  [client_2] Retrieved 2 chunks
  [client_4] Retrieved 1 chunks

Retrieved 5 documents from 3 clients: client_1, client_2, client_4

ANSWER:
The required training for new employees includes:

- New Hire Safety Orientation (first day)
- Annual Safety Refresher (all employees)
- Fire Extinguisher Training (annually)
- First Aid/CPR (for volunteers)
- Job-specific safety training

SOURCES:
   1. safety_manual.txt (from client_1)
   2. safety_manual.txt (from client_1)
   3. performance_management.txt (from client_2)




In [33]:
answer, docs = ask_question("How does the company handle confidential information?")

QUESTION: How does the company handle confidential information?

[Server] Query: 'How does the company handle confidential information?'
[Server] Routed to: client_6
[Server] Querying 1 clients...
  [client_6] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_6] Retrieved 5 chunks

Retrieved 5 documents from 1 clients: client_6

ANSWER:
The company handles confidential information through the following measures:

1. **Definition of Confidential Information**: Confidential information includes trade secrets, proprietary information, customer and vendor lists, financial data and business plans, product roadmaps and strategies, employee personal information, and source code and technical documentation.

2. **Obligations**:
   - Employees are required to maintain confidentiality during and after their employment.
   - All confidential materials must be returned upon termination of employment.
   - Employees are prohibited from discussing confident

## 7. Compare: With vs Without Routing

Let's compare performance with and without query routing.

In [34]:
import time

test_query = "What is the expense reimbursement policy for meals?"

print("Testing WITH routing enabled...\n")
start = time.time()
docs_with_routing = server.federated_retrieve(test_query)
time_with_routing = time.time() - start

clients_queried_with_routing = len(set(doc.get('client_id', 'unknown') for doc in docs_with_routing))

print(f"\nTime: {time_with_routing:.2f}s")
print(f"Clients queried: {clients_queried_with_routing}")
print(f"Documents retrieved: {len(docs_with_routing)}")

Testing WITH routing enabled...


[Server] Query: 'What is the expense reimbursement policy for meals?'
[Server] Routed to: client_7
[Server] Querying 1 clients...
  [client_7] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_7] Retrieved 5 chunks

Time: 0.59s
Clients queried: 1
Documents retrieved: 5


In [35]:
server.use_routing = False

print("Testing WITHOUT routing (broadcasting to all clients)...\n")
start = time.time()
docs_without_routing = server.federated_retrieve(test_query)
time_without_routing = time.time() - start

clients_queried_without_routing = len(set(doc.get('client_id', 'unknown') for doc in docs_without_routing))

print(f"\nTime: {time_without_routing:.2f}s")
print(f"Clients queried: {clients_queried_without_routing}")
print(f"Documents retrieved: {len(docs_without_routing)}")

server.use_routing = True

Testing WITHOUT routing (broadcasting to all clients)...


[Server] Query: 'What is the expense reimbursement policy for meals?'
[Server] Broadcasting to all 8 clients
[Server] Querying 8 clients...
  [client_1] Found 5 results
  [client_2] Found 5 results
  [client_3] Found 5 results
  [client_4] Found 5 results
  [client_5] Found 5 results
  [client_6] Found 5 results
  [client_7] Found 5 results
  [client_8] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_1] Retrieved 1 chunks
  [client_2] Retrieved 1 chunks
  [client_3] Retrieved 1 chunks
  [client_4] Retrieved 1 chunks
  [client_5] Retrieved 1 chunks

Time: 0.03s
Clients queried: 5
Documents retrieved: 5


## 7. Interactive Querying

In [40]:
my_question = "What is the leave policy?"

answer, docs = ask_question(my_question)

QUESTION: What is the leave policy?

[Server] Query: 'What is the leave policy?'
[Server] Routed to: client_2
[Server] Querying 1 clients...
  [client_2] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_2] Retrieved 5 chunks

Retrieved 5 documents from 1 clients: client_2

ANSWER:
The leave policy includes the following procedures and advance notice requirements:

**Procedure for Requesting Time Off:**
1. Access the employee intranet at https://intranet.acmecorp.com.
2. Navigate to the "Time Off" section.
3. Select the "Request Time Off" option.
4. Choose the dates and type of leave (vacation, personal, sick).
5. Add any relevant notes or comments.
6. Submit the request for manager approval.

**Advance Notice Requirements:**
- **Vacation (1-5 days):** 2 weeks advance notice.
- **Vacation (6+ days):** 4 weeks advance notice.
- **Personal days:** 1 week advance notice when possible.
- **Sick leave:** Notify the manager as soon as possible befor

## 8. Batch Queries - Routing Accuracy Analysis

Test routing accuracy across multiple domain-specific queries.

In [37]:
test_queries = [
    ("What is workplace safety policy?", "client_1"),
    ("How much vacation do employees get?", "client_2"),
    ("What are the product features?", "client_3"),
    ("What is the customer onboarding process?", "client_4"),
    ("What development methodology is used?", "client_5"),
    ("What are GDPR requirements?", "client_6"),
    ("How do I submit expenses?", "client_7"),
    ("What are the brand guidelines?", "client_8"),
]

results = []

for query, expected_client in test_queries:
    docs = server.federated_retrieve(query)
    routed_clients = list(set(doc.get('client_id', 'unknown') for doc in docs))
    
    results.append({
        "Query": query[:40] + "...",
        "Expected": expected_client,
        "Routed To": ", ".join(routed_clients),
        "Correct": "YES" if expected_client in routed_clients else "NO"
    })
    print(".", end="", flush=True)

print("\n\nRouting Accuracy Results:\n")
df_results = pd.DataFrame(results)
df_results


[Server] Query: 'What is workplace safety policy?'
[Server] Routed to: client_1, client_2
[Server] Querying 2 clients...
  [client_1] Found 5 results
  [client_2] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_1] Retrieved 3 chunks
  [client_2] Retrieved 2 chunks
.
[Server] Query: 'How much vacation do employees get?'
[Server] Routed to: client_2
[Server] Querying 1 clients...
  [client_2] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_2] Retrieved 5 chunks
.
[Server] Query: 'What are the product features?'
[Server] Routed to: client_3
[Server] Querying 1 clients...
  [client_3] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected clients...
  [client_3] Retrieved 5 chunks
.
[Server] Query: 'What is the customer onboarding process?'
[Server] Routed to: client_4
[Server] Querying 1 clients...
  [client_4] Found 5 results
[Server] Aggregating...
[Server] Fetching from selected 

,Query,Expected,Routed To,Correct
0,What is workplace safety policy?...,client_1,"client_2, client_1",YES
1,How much vacation do employees get?...,client_2,client_2,YES
2,What are the product features?...,client_3,client_3,YES
3,What is the customer onboarding process?...,client_4,client_4,YES
4,What development methodology is used?...,client_5,client_5,YES
5,What are GDPR requirements?...,client_6,client_6,YES
6,How do I submit expenses?...,client_7,client_7,YES
7,What are the brand guidelines?...,client_8,client_8,YES


## 9. System Configuration

View current system settings and configuration.

In [38]:
print("Current System Configuration:\n")
print(f"Routing Enabled: {server.use_routing}")
print(f"Max Clients per Query: {server.max_clients if server.use_routing else 'N/A (broadcasting)'}")
print(f"Aggregation Strategy: {config['retrieval']['aggregation_strategy'].upper()}")
print(f"Top-K per Client: {config['retrieval']['top_k_per_client']}")
print(f"Global Top-K: {config['retrieval']['global_top_k']}")
print(f"LLM Model: {config['llm']['model']}")
print(f"LLM Temperature: {config['llm']['temperature']}")

Current System Configuration:

Routing Enabled: True
Max Clients per Query: 3
Aggregation Strategy: RRF
Top-K per Client: 5
Global Top-K: 5
LLM Model: gpt-4o-mini
LLM Temperature: 0.7
